# Telco Customer Churn 데이터 탐색

목표: 컬럼 구조, 결측치, 타겟(`Churn`) 분포를 확인하고 SHAP 분석에 쓸 주요 특성을 추린다.

In [1]:
import pandas as pd

df = pd.read_csv("../data/raw/Telco-Customer-Churn.csv")
df.shape

(7043, 21)

In [2]:
df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [3]:
df.dtypes

customerID              str
gender                  str
SeniorCitizen         int64
Partner                 str
Dependents              str
tenure                int64
PhoneService            str
MultipleLines           str
InternetService         str
OnlineSecurity          str
OnlineBackup            str
DeviceProtection        str
TechSupport             str
StreamingTV             str
StreamingMovies         str
Contract                str
PaperlessBilling        str
PaymentMethod           str
MonthlyCharges      float64
TotalCharges            str
Churn                   str
dtype: object

In [4]:
df.isna().sum().sort_values(ascending=False)

customerID          0
gender              0
SeniorCitizen       0
Partner             0
Dependents          0
tenure              0
PhoneService        0
MultipleLines       0
InternetService     0
OnlineSecurity      0
OnlineBackup        0
DeviceProtection    0
TechSupport         0
StreamingTV         0
StreamingMovies     0
Contract            0
PaperlessBilling    0
PaymentMethod       0
MonthlyCharges      0
TotalCharges        0
Churn               0
dtype: int64

## 발견: `TotalCharges`는 숫자인데 문자열(object) 타입

`df.isna().sum()`으로는 결측치가 안 잡히지만, `TotalCharges`가 `object` dtype인 게 수상하다. 공백 문자열로 채워진 셀이 있는지 확인해본다 (보통 `tenure == 0`인 신규 고객).

In [5]:
blank_total_charges = df[df["TotalCharges"].astype(str).str.strip() == ""]
len(blank_total_charges), blank_total_charges[["tenure", "MonthlyCharges", "TotalCharges"]].head()

(11,
       tenure  MonthlyCharges TotalCharges
 488        0           52.55             
 753        0           20.25             
 936        0           80.85             
 1082       0           25.75             
 1340       0           56.05             )

→ 11개 행이 공백 문자열이고 전부 `tenure == 0`(막 가입한 고객이라 아직 청구된 총액이 없음). `common.py`는 컬럼명을 지정하지 않고, object 컬럼 중 숫자로 변환 가능한 비율이 높은 컬럼을 자동으로 숫자형으로 바꾸는 방식으로 이 문제를 일반적으로 처리한다 (`TotalCharges`를 직접 언급하지 않음).

In [6]:
df["Churn"].value_counts(normalize=True)

Churn
No     0.73463
Yes    0.26537
Name: proportion, dtype: float64

## 클래스 불균형

`Churn` = Yes 비율이 약 27%로 어느 정도 불균형이 있으나 HR 데이터셋만큼 심하지는 않음.

In [7]:
n = len(df)
id_like_cols = [c for c in df.columns if df[c].nunique() == n]
useful_cols = [c for c in df.columns if c not in ["Churn"] + id_like_cols]
categorical_like = [c for c in useful_cols if df[c].dtype == "object" and c != "TotalCharges"]
numeric_like = [c for c in useful_cols if c not in categorical_like]
print("ID성 컬럼:", id_like_cols)
print("범주형:", categorical_like)
print("수치형(TotalCharges 포함):", numeric_like)

ID성 컬럼: ['customerID']
범주형: []
수치형(TotalCharges 포함): ['gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'MonthlyCharges', 'TotalCharges']


→ `customerID`는 행마다 고유한 ID성 컬럼이라 자동 제거 대상.

## SHAP 분석에 쓸 주요 특성 5~8개

1. **Contract** — 계약 형태(월별/1년/2년), 이탈과 가장 강하게 연관된 것으로 잘 알려짐
2. **tenure** — 가입 기간
3. **MonthlyCharges** — 월 요금
4. **InternetService** — 인터넷 서비스 종류
5. **OnlineSecurity** — 온라인 보안 서비스 가입 여부
6. **TechSupport** — 기술지원 가입 여부
7. **PaymentMethod** — 결제 방식
8. **TotalCharges** — 총 청구 금액

`customerID`는 ID성이라 제외.